# 04 - Baseline Modeling & Benchmark
## Comparing Classical NLP Classifiers & Exporting Serialized Pipeline

In this notebook, we systematically train and benchmark classical models:
1. **Model Comparison**: Logistic Regression vs. Linear Support Vector Classifier (LinearSVC) vs. Multinomial/Complement Naive Bayes.
2. **Hyperparameter Tuning**: Regularization parameter $C$ optimization via Cross-Validation.
3. **Evaluation Suite**: Accuracy, Precision, Recall, F1-Score, Confusion Matrix, and ROC-AUC curve.
4. **End-to-End Pipeline Serialization**: Packaging the vectorizer and classifier into a single Scikit-learn `Pipeline` saved to `models/baseline_pipeline.pkl`.


In [ ]:
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)

# Load cleaned data
df = pd.read_csv("../data/processed/train_reviews_clean.csv")
X = df["clean_review"].fillna("")
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train size: {len(X_train)} | Test size: {len(X_test)}")


### 1. Negation-Preserving TF-IDF Vectorizer


In [ ]:
NEGATION_WORDS = {
    "not", "no", "never", "neither", "nor", "none", "nobody", "nowhere",
    "hardly", "scarcely", "barely", "cannot", "without", "against"
}
CUSTOM_STOP_WORDS = list(ENGLISH_STOP_WORDS - NEGATION_WORDS)

tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=3,
    max_df=0.85,
    stop_words=CUSTOM_STOP_WORDS
)


### 2. Benchmarking Multiple Classifiers


In [ ]:
models = {
    "Logistic Regression (C=2.5)": LogisticRegression(C=2.5, max_iter=1000, random_state=42),
    "LinearSVC (C=1.0)": LinearSVC(C=1.0, random_state=42, max_iter=2000),
    "Complement Naive Bayes": ComplementNB(alpha=1.0)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
comparison = {}

for name, clf in models.items():
    pipe = Pipeline([("tfidf", tfidf), ("clf", clf)])
    cv_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="accuracy", n_jobs=-1)
    pipe.fit(X_train, y_train)
    test_preds = pipe.predict(X_test)
    test_acc = accuracy_score(y_test, test_preds)
    
    comparison[name] = {
        "5-Fold CV Accuracy": f"{cv_scores.mean()*100:.2f}% (±{cv_scores.std()*100:.2f}%)",
        "Test Accuracy": f"{test_acc*100:.2f}%",
        "Test F1-Score": f"{f1_score(y_test, test_preds)*100:.2f}%"
    }

pd.DataFrame(comparison).T


### 3. Detailed Evaluation of the Best Model (Logistic Regression)


In [ ]:
best_pipeline = Pipeline([
    ("tfidf", tfidf),
    ("clf", LogisticRegression(C=2.5, max_iter=1000, random_state=42))
])

best_pipeline.fit(X_train, y_train)
y_pred = best_pipeline.predict(X_test)
y_prob = best_pipeline.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Negative", "Positive"]))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt=",d", cmap="Blues",
            xticklabels=["Negative", "Positive"],
            yticklabels=["Negative", "Positive"])
plt.title(f"Baseline Confusion Matrix (Acc: {accuracy_score(y_test, y_pred)*100:.2f}%)", fontsize=12)
plt.ylabel("Actual Label")
plt.xlabel("Predicted Label")
plt.tight_layout()
plt.show()


### 4. Serializing the Full Pipeline


In [ ]:
output_dir = Path("../models")
output_dir.mkdir(exist_ok=True)
pipeline_path = output_dir / "baseline_pipeline.pkl"

joblib.dump(best_pipeline, pipeline_path)
print(f"Serialized end-to-end pipeline saved to: {pipeline_path.resolve()}")
